# Primal and mixed elasticity methods

Compare displacement-only Galerkin, displacement–pressure GaLS and weak-symmetry
H(div) stress methods on explicit affine patches in 2D and 3D. BDM face modes
and zero-normal bubble enrichment are selected independently. Tensor RT stress
families are available in 2D.

The default data prescribe a nonzero displacement on the exterior boundary.
Each record contains separate displacement, pressure/stress/rotation errors
where applicable, spaces, traction convention and the original-equation residual.
Raw primal stress and mixed H(div) stress are different returned fields.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/f936d03d68415095808fc1f014bba11f4d06a2cefa1ab3c8035c72ad8126804d/introductory_methods-companion.zip"
COMPANION_SHA256 = "f936d03d68415095808fc1f014bba11f4d06a2cefa1ab3c8035c72ad8126804d"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("elasticity/introductory_methods.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)

import numpy as np


## Declare and solve primal elasticity equations

The user-written native operator is isotropic plane elasticity with
$\lambda_L=\mu_L=1$. An affine displacement gives a constant independently
known stress and zero body force. Three explicitly declared rigid modes form
the local kernel; translations and rotation have physical integral moments.

$$
\begin{aligned}
a_T(u,v)&=2(\varepsilon(u),\varepsilon(v))_T
 +(\operatorname{div}u,\operatorname{div}v)_T,\\
B_T\lambda&=\langle\lambda,v\rangle_{\partial T},\\
C_Tu&=-\langle u,\mu\rangle_{\partial T}.
\end{aligned}
$$

The P0 multiplier is the negative physical Cauchy traction with each fixed
macroface normal. The global RHS explicitly imposes negative displacement
moments on exterior traces; no global gauge is needed for these Dirichlet data.
The primary solve requires compatible native DOLFINx/UFL dependencies.


In [ ]:
import importlib.util
from pymhm import Equation, LocalEquations, MultiscaleProblem, assemble, columns, rows, compile_form
from pymhm.meshes.triangle import TriangleMesh
from pymhm.fem.traces.interval import FaceSpace, SkeletonSpace
from pymhm.fem.scalar.operators import boundary_data
from examples.variational_darcy import NativeFormProvider

native_available = importlib.util.find_spec("dolfinx") is not None
if native_available:
    import ufl
    from dolfinx import fem

    mesh = TriangleMesh.unit_square()
    trace_space = SkeletonSpace(mesh, components=2)
    gradient = np.array([[1.0, 2.0], [-1.0, 3.0]])
    offset = np.array([1.0, -2.0])
    stress = gradient + gradient.T + np.trace(gradient) * np.eye(2)

    def exact_displacement(points):
        """Return the affine displacement with independently known constant stress."""
        return points @ gradient.T + offset

    def local_forms(context):
        """Write isotropic elasticity and retain its declared three rigid modes."""
        space = fem.functionspace(context.space.mesh, ("Lagrange", 1, (2,)))
        u, v = ufl.TrialFunction(space), ufl.TestFunction(space)
        x = ufl.SpatialCoordinate(context.space.mesh)
        dx = ufl.dx(domain=context.space.mesh)
        points = space.tabulate_dof_coordinates()[:, :2].copy()
        center = points.mean(axis=0)
        rigid = np.zeros((2 * len(points), 3))
        rigid[0::2, 0], rigid[1::2, 1] = 1, 1
        rigid[0::2, 2], rigid[1::2, 2] = -(points[:, 1] - center[1]), points[:, 0] - center[0]
        rotation = ufl.as_vector((-x[1] + float(center[1]), x[0] - float(center[0])))
        return LocalEquations(
            (2 * ufl.inner(ufl.sym(ufl.grad(u)), ufl.sym(ufl.grad(v))) + ufl.div(u) * ufl.div(v))
            * dx,
            ufl.inner(ufl.as_vector((0.0, 0.0)), v) * dx,
            columns(
                *(
                    float(sign) * v[j] * context.ds(side + 1)
                    for side, sign in enumerate(context.signs)
                    for j in range(2)
                )
            ),
            rows(
                *(
                    -float(sign) * u[j] * context.ds(side + 1)
                    for side, sign in enumerate(context.signs)
                    for j in range(2)
                )
            ),
            context.trace_dofs,
            kernel=rigid,
            moments=columns(v[0] * dx, v[1] * dx, ufl.inner(rotation, v) * dx),
            metadata={"points": points},
        )

    boundary, _ = boundary_data(trace_space, exact_displacement)
    problem = MultiscaleProblem(
        Equation(0, -np.r_[boundary, np.zeros(3 * len(mesh.cells))]),
        NativeFormProvider(mesh, trace_space, local_forms, 2),
        range(len(mesh.cells)),
        trace_space.size,
        (3,) * len(mesh.cells),
    )
    system = assemble(problem)
    solution = system.solve()
    errors = [
        float(np.max(np.abs(field.reshape(-1, 2) - exact_displacement(record["points"]))))
        for field, record in zip(solution.fields, system.local_metadata, strict=True)
    ]
    expected_trace = np.concatenate([-stress @ normal for normal in mesh.normals])
    np.testing.assert_allclose(solution.trace, expected_trace, atol=1e-10, rtol=0)
    assert max(errors) < 1e-10
    print(
        {
            "native_available": True,
            "spaces": "P1 displacement / P0 vector trace / 3 rigid modes",
            "maximum_displacement_nodal_error_by_cell": errors,
            "global_original_relative_residual": solution.raw_residual,
        }
    )
else:
    print(
        {
            "native_available": False,
            "status": "User-written UFL primary solve not executed: select the Pixi fem kernel.",
        }
    )


## Compare the supported methods
These comparisons invoke the existing formulation implementations through the application catalogue and preserve their numerical records. The user-defined primary UFL solve above exercises the generic local/global API; the catalogue drivers remain compatibility comparisons.


In [ ]:
from examples.tutorial_vector_variants import run_variant

selected_methods = (
    "elasticity-primal-2d",
    "elasticity-primal-3d",
    "elasticity-gals-2d",
    "elasticity-gals-3d",
    "elasticity-bdm-2d",
    "elasticity-bdm-plus-2d",
    "elasticity-bdm-plusplus-2d",
    "elasticity-bdm-3d",
    "elasticity-rt-2d",
    "elasticity-rt-plus-2d",
)
records = [run_variant(name) for name in selected_methods]
for record in records:
    print(record["variant"], record["errors"], record["provenance"])


## Homogeneous data and the incompressible mixed limit

The primal method requires a finite Lamé parameter. The mixed incompressible
patch uses a solenoidal affine displacement and physical mean pressure $2.3$.
These patches introduce the APIs; they do not replace convergence studies.


In [ ]:
homogeneous = run_variant("elasticity-primal-2d", homogeneous=True)
incompressible = run_variant("elasticity-gals-2d", incompressible=True)
print(homogeneous)
print(incompressible)
